Notebook to set up the first pipe line for a speech-to-speech translation. 

In [24]:
from transformers import pipeline
import torch 

In [25]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch.cuda.is_available()

True

In [26]:
classifier = pipeline(
    "audio-classification", model="MIT/ast-finetuned-speech-commands-v2", device=device
)

Loading weights: 100%|██████████| 203/203 [00:00<00:00, 13607.64it/s]


In [27]:
classifier.model.config.id2label

{0: 'backward',
 1: 'follow',
 2: 'five',
 3: 'bed',
 4: 'zero',
 5: 'on',
 6: 'learn',
 7: 'two',
 8: 'house',
 9: 'tree',
 10: 'dog',
 11: 'stop',
 12: 'seven',
 13: 'eight',
 14: 'down',
 15: 'six',
 16: 'forward',
 17: 'cat',
 18: 'right',
 19: 'visual',
 20: 'four',
 21: 'wow',
 22: 'no',
 23: 'nine',
 24: 'off',
 25: 'three',
 26: 'left',
 27: 'marvin',
 28: 'yes',
 29: 'up',
 30: 'sheila',
 31: 'happy',
 32: 'bird',
 33: 'go',
 34: 'one'}

In [28]:
from transformers.pipelines.audio_utils import ffmpeg_microphone_live


def launch_fn(
    wake_word="marvin",
    prob_threshold=0.5,
    chunk_length_s=2.0,
    stream_chunk_s=0.25,
    debug=False,
):
    if wake_word not in classifier.model.config.label2id.keys():
        raise ValueError(
            f"Wake word {wake_word} not in set of valid class labels, pick a wake word in the set {classifier.model.config.label2id.keys()}."
        )

    sampling_rate = classifier.feature_extractor.sampling_rate

    mic = ffmpeg_microphone_live(
        sampling_rate=sampling_rate,
        chunk_length_s=chunk_length_s,
        stream_chunk_s=stream_chunk_s,
    )


    for chunk in mic:
        # Ignore the 0.25 s temporary pieces.
        if chunk["partial"]:
            continue

        # This will be approximately 2 seconds at 16 kHz.
        print("Classifying chunk:", chunk["raw"].shape)

        prediction = classifier(chunk)[0]

        if debug:
            print(prediction)

        if (
            prediction["label"] == wake_word
            and prediction["score"] > prob_threshold
        ):
            print("Wake word detected.")
            return True

In [29]:
transcriber = pipeline(
    "automatic-speech-recognition", model="openai/whisper-small.en", device=device
)

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 12474.60it/s]


In [30]:
import sys

In [35]:
def transcribe_audio(chunk_length_s=3.0, stream_chunk_s=1.0):
    sampling_rate = transcriber.feature_extractor.sampling_rate

    mic = ffmpeg_microphone_live(
        sampling_rate=sampling_rate,
        chunk_length_s=chunk_length_s,
        stream_chunk_s=stream_chunk_s
    )

    print("Start speaking bsdk...")

    for chunk in mic: 
        if chunk["partial"]: 
            continue

        result = transcriber(chunk  )

        print(f"Transcription result bsdk: {result}")

        text = result["text"]

        sys.stdout.write("\r\033[K" + text)
        sys.stdout.flush()
        return text


   

In [10]:
# Undo the incompatible language/task configuration.
transcriber.model.generation_config.language = None
transcriber.model.generation_config.task = None

# Keep output bounded without passing generate_kwargs on every call.
transcriber.model.generation_config.max_new_tokens = 64

# to print ts just do transcribe_audio() lol 

In [11]:
from huggingface_hub import get_token
import requests

In [12]:
import os 

API_URL = "https://router.huggingface.co/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
}

def query(payload):
    if "messages" not in payload:
        payload= build_prompt(payload)
    response = requests.post(API_URL, headers=headers, json=payload)
    data = response.json()
    print(data)
    return data["choices"][0]["message"]["content"]

 

In [13]:
def build_prompt(user_message):
    return {
        "messages": [
            {
                "role": "user",
                "content": user_message
            }
        ],
        "model": "Qwen/Qwen2.5-32B-Instruct:featherless-ai"
    } 

In [17]:
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan, set_seed
import torch

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

inputs = processor(text="Hello, my dog is cute", return_tensors="pt")
speaker_embeddings = torch.zeros((1, 512))  # or load xvectors from a file

set_seed(555)  # make deterministic

# generate speech
speech = model.generate(inputs["input_ids"], speaker_embeddings=speaker_embeddings, vocoder=vocoder)
speech.shape

Loading weights: 100%|██████████| 396/396 [00:00<00:00, 45221.61it/s]
[transformers] SpeechT5ForTextToSpeech LOAD REPORT from: microsoft/speecht5_tts
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
speecht5.encoder.prenet.encode_positions.pe | UNEXPECTED |  | 
speecht5.decoder.prenet.encode_positions.pe | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] You are using a model of type `hifigan` to instantiate a model of type `speecht5_hifigan`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
Loading weights: 100%|██████████| 158/158 [00:00<00:00, 58279.84it/s]


torch.Size([15872])

In [18]:
from IPython.display import Audio

Audio(speech, rate=16000)

In [1]:
from datasets import load_dataset

d:\Code\Learning\audio-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors",revision="refs/convert/parquet", split="validation") 
speaker_embeddings = embeddings_dataset[7306]["xvector"]
speaker_embeddings = torch.tensor(speaker_embeddings).unsqueeze(0)

In [22]:
def synthesise(text):
    inputs = processor(text=text, return_tensors="pt")
    speaker_embeddings = torch.zeros((1, 512))  # or load xvectors from a file

    set_seed(555)  # make deterministic

    # generate speech
    speech = model.generate(inputs["input_ids"], speaker_embeddings=speaker_embeddings, vocoder=vocoder)
    return speech

In [23]:
from IPython.display import Audio

audio = synthesise(
    "Hugging Face is a company that provides natural language processing and machine learning tools for developers.",
)

Audio(audio, rate=16000)

In [33]:
launch_fn()


Using microphone: Headset Microphone (2- DualSense Wireless Controller)
Classifying chunk: (32000,)
Wake word detected.


True

In [37]:
launch_fn()
transcription = transcribe_audio()
response = query(transcription)
audio = synthesise(response)

Audio(audio, rate=16000, autoplay=True)

Using microphone: Headset Microphone (2- DualSense Wireless Controller)
Classifying chunk: (32000,)
Classifying chunk: (32000,)
Wake word detected.
Start speaking bsdk...
Using microphone: Headset Microphone (2- DualSense Wireless Controller)
Transcription result bsdk: {'text': ' I know I had a question.', 'partial': [False]}
 I know I had a question.{'id': '64066e59-2c9f-494b-81d6-0a3a81b93cf7', 'object': 'chat.completion', 'created': 1784770197, 'model': 'Qwen/Qwen2.5-32B-Instruct', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "It seems like you might be thinking of a question but aren't sure what it is right now. If you need help remembering your question or if there's something specific you'd like to know, feel free to share any details you have, and I'll do my best to assist you!"}, 'finish_reason': 'stop'}], 'system_fingerprint': 'fp1-rss-h1', 'usage': {'prompt_tokens': 36, 'completion_tokens': 58, 'total_tokens': 94}}
